# Stage 3 · Finalize the dataset

Join accepted Stage 2 transcripts with their Stage 1 records and package the BTC scenarios and audits in one ZIP.

In [ ]:
# Configure
from pathlib import Path
from runpy import run_path
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZipFile
import sys

BATCH_SIZE = 20
KAGGLE_INPUT = Path('/kaggle/input')
INPUT_ARCHIVE = None

if KAGGLE_INPUT.is_dir():
    stages = [helper.parents[1] for helper in KAGGLE_INPUT.rglob('stage-3/src/finalize.py')]
    if stages:
        if len(stages) != 1:
            raise ValueError('Attach only one Stage 3 input dataset.')
        STAGE = stages[0]
    else:
        input_names = ('stage3-m1-input.zip', 'stage-3-kaggle-input.zip')
        archives = [path for name in input_names for path in KAGGLE_INPUT.rglob(name)]
        if len(archives) != 1:
            raise ValueError('Attach one Stage 3 Kaggle input dataset.')
        INPUT_ARCHIVE = archives[0]
    OUTPUT = Path('/kaggle/working/stage-3-output.zip')
else:
    STAGE = Path.cwd().resolve()
    if STAGE.name != 'stage-3':
        STAGE = STAGE / 'stage-3'
    OUTPUT = STAGE / 'output' / 'stage-3-output.zip'
    local_archive = STAGE / 'stage3-m1-input.zip'
    if local_archive.is_file():
        INPUT_ARCHIVE = local_archive

if INPUT_ARCHIVE is not None:
    extracted = TemporaryDirectory(prefix='stage-3-input-')
    with ZipFile(INPUT_ARCHIVE) as archive:
        archive.extractall(extracted.name)
    STAGE = Path(extracted.name) / 'stage-3'

ROOT = STAGE.parent
INPUT = STAGE / 'input'
sys.path.insert(0, str(STAGE / 'src'))
finalize = run_path(str(STAGE / 'src/finalize.py'))['finalize']

In [ ]:
# Finalize
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(prefix='.stage-3-zip-', dir=OUTPUT.parent) as temporary:
    staging = Path(temporary)
    scenarios = staging / 'scenarios'
    stats = finalize(
        INPUT, scenarios, batch_size=BATCH_SIZE, btc_dir=ROOT / 'btc',
        venue_catalog=ROOT / 'stage-1/input/catalog/public-delivery-addresses.json',
    )
    staged_zip = staging / OUTPUT.name
    with ZipFile(staged_zip, 'w', compression=ZIP_DEFLATED) as archive:
        for path in sorted(scenarios.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(scenarios).as_posix())
    staged_zip.replace(OUTPUT)

In [ ]:
# Review
print(f"{stats['tier']}: {stats['total_journeys']} journeys, {stats['total_sessions']} sessions")
print(f"Duplicates removed: {stats['duplicates_removed']} · Privacy rejections: {stats['privacy_rejections']}")
print(f"Output ZIP: {OUTPUT}")